In [3]:
from transformers import pipeline

ner = pipeline("ner", grouped_entities=True)
results = ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")
print(type(results))
print(results)

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Hardware accelerator e.g. GPU is

<class 'list'>
[{'entity_group': 'PER', 'score': 0.9981694, 'word': 'Sylvain', 'start': 11, 'end': 18}, {'entity_group': 'ORG', 'score': 0.9796019, 'word': 'Hugging Face', 'start': 33, 'end': 45}, {'entity_group': 'LOC', 'score': 0.9932106, 'word': 'Brooklyn', 'start': 49, 'end': 57}]


In [9]:
print( "Type: ", results[0]['entity_group'], " Name: ", results[0]['word'], " Score: ", results[0]['score'])

Type:  PER  Name:  Sylvain  Score:  0.9981694


In [7]:
for e in results:
    print(e['entity_group'] ," :", e['word'])

PER  : Sylvain
ORG  : Hugging Face
LOC  : Brooklyn


In [ ]:
from pymongo import MongoClient
from openai import OpenAI
from transformers import pipeline
import pandas as pd
import re
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
client = MongoClient("192.168.2.70", 27017)
mydb = client["smsdata"]
mycol = mydb["smsmessages"]

client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [45]:


def find_word_after_from(text):
    # Regular expression to match the word after **FROM**
    match = re.search(r'\*\*.*?\*\*(.*?)\n', text)
    
    if match:
        # Return the word found after **FROM**
        return match.group(1)
    else:
        return None  # Return None if no match is found

results = mycol.find({'type':1,'predict':1})
for doc in results:
    
    #print(doc['message'])
    result = find_word_after_from(doc['message'])
    print("From: ", result.lstrip())  # remove leading whitespace
    
    #ner = pipeline("ner", grouped_entities=True)
    #entities = ner(doc['message'])
    #for x in entities:
        #print(x)
    break


From:  City Travels  


In [39]:
import re

def extract_after_asterisk_pattern(text):
    # Regular expression to match text starting and ending with '**' and capture up to the newline
    match = re.search(r'\*\*.*?\*\*(.*?)\n', text)
    
    if match:
        # Return the content captured after '**' up to the newline
        return match.group(1).strip()  # Strip to remove leading/trailing whitespace
    else:
        return None  # Return None if no match is found

# Example usage:
text = """Some other text
before **START** and this is what I want to capture.
More text after."""

result = extract_after_asterisk_pattern(text)
print(result)


and this is what I want to capture.


In [ ]:
import re
from transformers import pipeline
from pymongo import MongoClient
from openai import OpenAI
import pandas as pd
from py2neo import Graph
import os
from dotenv import load_dotenv

load_dotenv()

client = MongoClient("192.168.2.70", 27017)
mydb = client["messages"]
mycol = mydb["smsmessages"]

graph = Graph(os.getenv("NEO4J_URI"), auth=(os.getenv("NEO4J_USER"), os.getenv("NEO4J_PASSWORD")))

client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))


def find_text_after_asterisks_until_newline(text):
    # Regular expression to match text following **pattern** up to a newline
    pattern = r'\*\*(.*?)\*\*(.*?)(?=\n|$)'
    matches = re.findall(pattern, text)
    # Extract only the part after the **...** until the newline
    results = [match[1].strip() for match in matches]
    return results

def find_money(text):
    # Regular expression to match cash amounts
    pattern = r'\$\d{1,3}(?:,\d{3})*(?:\.\d{2})?'
    matches = re.findall(pattern, text)
    return matches


results = mycol.find({'type':2,'predict':2})
count = 0
badmatch = 0
maxCharlength = 30

ner = pipeline("ner", grouped_entities=True)


f = open("financesms.txt", "w")

msgCount = 1000  # do not duplicate numbers for travel messages
for doc in results:
    
    msgCount = msgCount + 1
    
    f.write("*** Start of Message ***\n\n")
    f.write(doc['message'])
    f.write("\n\n*** End of Message*** \n\n\n\n")
    
    #print(doc['message'])
    result = find_text_after_asterisks_until_newline(doc['message'])
    #print("From: ", result.lstrip())  # remove leading whitespace
    try:
        if(len(result[0]) > maxCharlength or len(result[1]) >maxCharlength ):
            badmatch = badmatch + 1
            continue
        fromSource = result[0].lstrip()
        toSource = result[1].lstrip()
        # Check for Empty String
        if(fromSource ==""):
            fromSource = "Empty"
        if(toSource == ""):
            toSource = "Empty"
        print("From: ",fromSource, "To: ", toSource)
    except:
        #print(doc['message'])
        badmatch = badmatch +1
        
    # Get money amounts if present
    money_matches = find_money(doc['message'])
    for money in money_matches:
        cypher = f'MERGE(m:Finance:SMS:Message {{id: {msgCount}}}) MERGE(f:Money {{amount:"{money}"}}) MERGE((m) - [:funds {{date:"1/4/2020"}}] -> (f))'
        graph.run(cypher)
    
    entities = ner(doc['message'])
    for x in entities:
        toFlag = False
        fromFlag = False
        
        #print(x['entity_group'], ": ", x['word'], " with score: ", x['score'])
        
        if(x['word'] == toSource):
            toFlag = True
        elif(x['word'] == fromSource):
            fromFlag = True

        if(x['score'] < .70): # Throw away poor results
            continue
        name = x["word"]
        if(x['entity_group'] == "LOC"):  # Add location
            cypher = f'MERGE(m:Finance:SMS:Message {{id: {msgCount}}}) MERGE(l:Location {{name:"{name}"}}) MERGE((m) - [:at {{date:"1/4/2020"}}] -> (l))'
            graph.run(cypher)
        if(x['entity_group'] == "ORG"):
            #MERGE(m:Message {id: 1}) MERGE(l:Location {name:"jimmy"}) MERGE((m) - [:at {date:"1/4/2020"}] -> (l))
            name = x["word"]
            cypher = f'MERGE(m:Finance:SMS:Message {{id: {msgCount}}}) MERGE(o:Organization {{name:"{name}"}}) MERGE((m) - [:associated {{date:"1/4/2020"}}] -> (o))'
            graph.run(cypher)
            if(toFlag or fromFlag):
                relationship = 'received' if (toFlag == True) else 'sent'
                cypher = f'MERGE(m:Finance:SMS:Message {{id: {msgCount}}}) MERGE(o:Organization {{name:"{name}"}}) MERGE((m) - [:{relationship} {{date:"1/4/2020"}}] -> (o))'
                graph.run(cypher)
            
        if(x['entity_group'] == "PER"):
            cypher = f'MERGE(m:Finance:SMS:Message {{id: {msgCount}}}) MERGE(p:Person {{name:"{name}"}}) MERGE((m) - [:related {{date:"1/4/2020"}}] -> (p))'
            graph.run(cypher)
            if(toFlag or fromFlag):
                relationship = 'received' if (toFlag == True) else 'sent'
                cypher = f'MERGE(m:Finance:SMS:Message {{id: {msgCount}}}) MERGE(p:Person {{name:"{name}"}}) MERGE((m) - [:{relationship} {{date:"1/4/2020"}}] -> (p))'
                graph.run(cypher)        
                
    #if(msgCount >10):
        #break
        
print("Total Count: ", count, " NoFromTo: ", badmatch)
f.close()